# Cross-Validation

## Learning Objectives
1. Implement k-fold cross-validation from scratch using only NumPy.
2. Compare KFold, StratifiedKFold, TimeSeriesSplit, and GroupKFold in sklearn.
3. Apply nested cross-validation for unbiased hyperparameter tuning.
4. Quantify cross-validation variance to detect unstable model estimates.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import (KFold, StratifiedKFold,
                                     TimeSeriesSplit, GroupKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.metrics import r2_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: K-Fold Cross-Validation from Scratch (NumPy)

In [ ]:
def kfold_indices(n, k, shuffle=True, seed=42):
    # Yield (train_idx, val_idx) tuples for k-fold CV.
    idx = np.arange(n)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(idx)
    fold_sizes = np.full(k, n // k)
    fold_sizes[:n % k] += 1
    current = 0
    for size in fold_sizes:
        val_idx   = idx[current : current + size]
        train_idx = np.concatenate([idx[:current], idx[current + size:]])
        yield train_idx, val_idx
        current += size

X_cv, y_cv = make_classification(n_samples=300, n_features=10, random_state=42)

def fit_eval_cv(X_tr, y_tr, X_va, y_va):
    clf = LogisticRegression(max_iter=500)
    clf.fit(X_tr, y_tr)
    return (clf.predict(X_va) == y_va).mean()

scores = []
for tr_idx, va_idx in kfold_indices(len(X_cv), k=5):
    score = fit_eval_cv(X_cv[tr_idx], y_cv[tr_idx], X_cv[va_idx], y_cv[va_idx])
    scores.append(score)

print(f"5-fold CV scores: {[round(s, 4) for s in scores]}")
print(f"Mean accuracy:  {np.mean(scores):.4f} +/- {np.std(scores):.4f}")
print("K-fold splits data into k equally-sized folds, using each as validation once.")


## Level 2: KFold / StratifiedKFold / TimeSeriesSplit / GroupKFold (sklearn)

In [ ]:
# Four CV strategies, each designed for a specific data structure.

X_cls, y_cls = make_classification(n_samples=400, n_features=12,
                                    weights=[0.7, 0.3], random_state=7)
groups = np.repeat(np.arange(40), 10)   # 40 patient groups of 10 samples each

# Temporally correlated data for TimeSeriesSplit
time_X = np.cumsum(np.random.randn(400, 12), axis=0)
time_y = (time_X[:, 0] > 0).astype(int)

strategies = {
    "KFold(shuffle)": (KFold(n_splits=5, shuffle=True, random_state=42), X_cls, y_cls, None),
    "StratifiedKFold": (StratifiedKFold(n_splits=5, shuffle=True, random_state=42), X_cls, y_cls, None),
    "TimeSeriesSplit": (TimeSeriesSplit(n_splits=5), time_X, time_y, None),
    "GroupKFold":      (GroupKFold(n_splits=5), X_cls, y_cls, groups),
}

clf_base = LogisticRegression(max_iter=500, random_state=42)
try:
    for name, (cv, X, y, g) in strategies.items():
        s = cross_val_score(clf_base, X, y, cv=cv, groups=g, scoring="accuracy", n_jobs=1)
        print(f"{name:20s}  mean={s.mean():.4f}  std={s.std():.4f}")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM -- reduce dataset size")
        torch.cuda.empty_cache()
    raise

print("StratifiedKFold: preserves class ratio per fold -- use for imbalanced data.")
print("TimeSeriesSplit: no future-to-past leakage -- mandatory for forecasting.")
print("GroupKFold: no cross-group contamination -- use for patient/user data.")


## Real-World Example 1: Nested Cross-Validation

In [ ]:
# Nested CV: outer loop estimates generalisation error; inner loop tunes hyperparams.
# Non-nested CV with tuning is optimistic (positively biased).

X_nest, y_nest = make_classification(n_samples=300, n_features=10, random_state=1)

pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC())])
param_grid = {"clf__C": [0.1, 1.0, 10.0], "clf__kernel": ["rbf", "linear"]}

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# Outer loop provides the honest generalisation estimate
outer_scores = []
for train_idx, test_idx in outer_cv.split(X_nest):
    X_tr, X_te = X_nest[train_idx], X_nest[test_idx]
    y_tr, y_te = y_nest[train_idx], y_nest[test_idx]
    grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring="accuracy", n_jobs=1)
    grid.fit(X_tr, y_tr)
    outer_scores.append(grid.score(X_te, y_te))

# Non-nested (optimistic): tune and evaluate on the same outer folds
non_nested = cross_val_score(GridSearchCV(pipe, param_grid, cv=inner_cv, n_jobs=1),
                              X_nest, y_nest, cv=outer_cv, n_jobs=1)

print(f"Nested CV accuracy:     {np.mean(outer_scores):.4f} +/- {np.std(outer_scores):.4f}")
print(f"Non-nested CV accuracy: {non_nested.mean():.4f} +/- {non_nested.std():.4f}")
print(f"Optimism bias:          {non_nested.mean() - np.mean(outer_scores):+.4f}")
print("Nested CV is the gold standard for model selection + evaluation.")


## Real-World Example 2: Temporal Cross-Validation for Time Series

In [ ]:
# TimeSeriesSplit never leaks future data into training.
# Standard KFold randomly shuffles, causing data-from-the-future leakage.

np.random.seed(42)
T = 500
y_ts = np.zeros(T)
for i in range(1, T):
    y_ts[i] = 0.8 * y_ts[i-1] + np.random.randn() * 0.5

def make_lag_features(y, lags=5):
    n = len(y)
    X = np.column_stack([y[i:n-lags+i] for i in range(lags)])
    return X, y[lags:]

X_ts, y_ts_cut = make_lag_features(y_ts, lags=5)
ridge = Ridge(alpha=1.0)
tss   = TimeSeriesSplit(n_splits=5)
kf    = KFold(n_splits=5, shuffle=True, random_state=42)

ts_scores = cross_val_score(ridge, X_ts, y_ts_cut, cv=tss, scoring="r2")
kf_scores  = cross_val_score(ridge, X_ts, y_ts_cut, cv=kf,  scoring="r2")

print(f"TimeSeriesSplit R2: {ts_scores.mean():.4f} +/- {ts_scores.std():.4f}  (honest)")
print(f"Standard KFold  R2: {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}  (leaks future)")
print(f"Leakage inflation:  {kf_scores.mean() - ts_scores.mean():+.4f}")
print("For time-series: always use TimeSeriesSplit or walk-forward validation.")


## Real-World Example 3: CV Variance Analysis

In [ ]:
# High CV variance indicates: small dataset, wrong strategy, or unstable model.
# Track coefficient of variation (std/mean) to detect instability.

X_var, y_var = make_classification(n_samples=200, n_features=8, random_state=5)
cv_var = KFold(n_splits=10, shuffle=True, random_state=42)

models_var = {
    "LogisticRegression": LogisticRegression(max_iter=500),
    "DecisionTree(d=3)":  DecisionTreeClassifier(max_depth=3, random_state=42),
    "DecisionTree(full)": DecisionTreeClassifier(random_state=42),   # likely overfit
    "RandomForest":       RandomForestClassifier(n_estimators=50, random_state=42),
}

print(f"{'Model':<22}  {'Mean Acc':>9}  {'Std':>6}  {'CV% (std/mean)':>15}")
for name, mdl in models_var.items():
    s = cross_val_score(mdl, X_var, y_var, cv=cv_var, scoring="accuracy", n_jobs=1)
    cv_pct = (s.std() / s.mean()) * 100
    print(f"{name:<22}  {s.mean():>9.4f}  {s.std():>6.4f}  {cv_pct:>15.1f}%")

print()
print("High CV% --> model is unstable; consider more data, regularisation, or ensembling.")
print("Fully-grown trees have high variance; RandomForest reduces it via averaging.")
